# Predicción de recompra en un e-commerce argentino
### Pre-entrega 2 · Ingesta, radiografía y funciones de transformación
**Data Science I · Coderhouse** · Luciano Rivero · Octubre 2026

---

**Contexto de negocio.** Un e-commerce argentino de consumo masivo vende por tienda propia, Mercado Libre, Instagram Shop y WhatsApp. La pregunta que guía el proyecto es: *¿qué clientes tienen alta probabilidad de volver a comprar en los próximos 90 días, y en cuáles conviene invertir para retenerlos?* Responderla permite dirigir el presupuesto de retención (descuentos, remarketing, envío gratis) solo a quienes lo necesitan, sin regalar margen a quienes iban a volver igual.

**Dataset.** `ventas_ds.csv`: 7.617 compras entre enero 2025 y junio 2026. Es un dataset sintético generado con Python (semilla 42) con lógica de negocio realista: inflación mensual, Hot Sale y CyberMonday, suba de comisión de Mercado Libre. Una fila = una compra, con el estado del cliente en ese momento. Variable objetivo: `recompra_90d` (clasificación binaria). Fuente, script generador y diccionario de datos en el [repositorio de GitHub](https://github.com/riveroluciano97-art/ecommerce-ar-data-science).

**Objetivo de este notebook.** Hacer el diagnóstico técnico inicial del dataset y programar las primeras funciones de transformación:

1. Ingesta de datos y primer vistazo
2. Radiografía técnica (dimensiones, tipos de datos, estadística descriptiva)
3. Diagnóstico de datos faltantes
4. Lógica algorítmica y funciones personalizadas (`def` + `lambda`)
5. Conclusiones y próximos pasos

## 0. Setup

In [1]:
# ==========================================
# SETUP
# ==========================================
import pandas as pd

# Muestro todas las columnas (son 23): sin esto pandas esconde las del medio con "..."
pd.set_option('display.max_columns', None)

## 1. Ingesta de datos y primer vistazo

Cargo el CSV directamente desde el link *raw* del repositorio público. Así el notebook corre igual en Google Colab o en Jupyter local, sin subir archivos a mano, y cualquiera puede reproducirlo.

In [2]:
# ==========================================
# BLOQUE 1: INGESTA DE DATOS
# ==========================================
URL_DATASET = ('https://raw.githubusercontent.com/riveroluciano97-art/'
               'ecommerce-ar-data-science/main/data/ventas_ds.csv')

df = pd.read_csv(URL_DATASET)

# display() renderiza la tabla en HTML con formato (más legible que print)
display(df.head())

,id_venta,fecha,id_cliente,provincia,fecha_alta_cliente,antiguedad_dias,compras_previas,dias_desde_ultima_compra,gasto_previo_acum,canal,comision_canal_pct,categoria,producto,precio_unitario,costo_unitario,cantidad,monto,descuento_pct,medio_pago,cuotas,es_evento,indice_inflacion,recompra_90d
0,1,2025-01-01,335,CABA,2025-01-01,0,0,NaN,0.00,Instagram Shop,0.08,Indumentaria,Vestido midi,56018.26,30959.05,1,56018.26,0.00,Tarjeta de débito,1,False,1.045,1.0
1,2,2025-01-01,455,Buenos Aires,2025-01-01,0,0,NaN,0.00,Instagram Shop,0.08,Hogar,Set sábanas,40803.13,24461.18,1,40803.13,0.00,Billetera virtual,1,False,1.045,0.0
2,3,2025-01-01,1185,Buenos Aires,2024-03-10,297,4,39.0,330802.50,Instagram Shop,0.08,Belleza,Esmalte gel,20291.41,9984.82,3,60874.23,0.00,Tarjeta de crédito,6,False,1.045,1.0
3,4,2025-01-01,1913,Mendoza,2024-09-07,116,2,17.0,156711.14,Tienda propia,0.06,Indumentaria,Buzo canguro,27431.98,14441.13,2,54863.95,0.00,Transferencia,1,False,1.045,1.0
4,5,2025-01-01,2286,Buenos Aires,2025-01-01,0,0,NaN,0.00,Mercado Libre,0.14,Indumentaria,Campera puffer,59662.55,32486.89,1,56679.43,0.05,Billetera virtual,1,False,1.045,0.0


**Primera lectura.** Cada fila es una compra con tres grupos de información: el estado del cliente en ese momento (`antiguedad_dias`, `compras_previas`, `dias_desde_ultima_compra`, `gasto_previo_acum`), los datos de la compra (canal, producto, precio, descuento, medio de pago) y el target `recompra_90d`. Ya se ve un patrón: en las filas 0, 1 y 4 `compras_previas` es 0 y `dias_desde_ultima_compra` es `NaN`. Son primeras compras, así que no existe una compra anterior desde la cual medir la recencia. Lo verifico en el bloque 3.

## 2. Radiografía técnica del dataset

### 2.1 Dimensiones y duplicados

In [3]:
# ==========================================
# BLOQUE 2: RADIOGRAFÍA TÉCNICA
# ==========================================
# shape devuelve una tupla (filas, columnas): la desempaqueto en dos variables
filas, columnas = df.shape
print(f'Dimensiones del dataset: {df.shape}')
print(f'-> {filas:,} filas (compras) y {columnas} columnas')
print(f'-> {df["id_cliente"].nunique():,} clientes únicos')

# Duplicados: una venta registrada dos veces inflaría la facturación y la tasa de recompra
print(f'-> Filas duplicadas completas: {df.duplicated().sum()}')
print(f'-> id_venta repetidos: {df.duplicated(subset="id_venta").sum()}')

Dimensiones del dataset: (7617, 23)
-> 7,617 filas (compras) y 23 columnas
-> 4,208 clientes únicos
-> Filas duplicadas completas: 0
-> id_venta repetidos: 0


El dataset tiene **7.617 filas y 23 columnas** (supera los mínimos de 5.000 filas y 12 columnas) con 4.208 clientes distintos: en promedio, 1,8 compras por cliente en el período. No hay filas duplicadas ni ventas registradas dos veces.

### 2.2 Tipos de datos

In [4]:
# info() muestra el tipo de cada columna y cuántos valores no nulos tiene
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7617 entries, 0 to 7616
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_venta                  7617 non-null   int64  
 1   fecha                     7617 non-null   object 
 2   id_cliente                7617 non-null   int64  
 3   provincia                 7617 non-null   object 
 4   fecha_alta_cliente        7617 non-null   object 
 5   antiguedad_dias           7617 non-null   int64  
 6   compras_previas           7617 non-null   int64  
 7   dias_desde_ultima_compra  3863 non-null   float64
 8   gasto_previo_acum         7617 non-null   float64
 9   canal                     7617 non-null   object 
 10  comision_canal_pct        7617 non-null   float64
 11  categoria                 7617 non-null   object 
 12  producto                  7617 non-null   object 
 13  precio_unitario           7617 non-null   float64
 14  costo_un

In [5]:
# ¿Hay números o fechas "disfrazados" de texto? Recorro cada columna no numérica
# y muestro cuántos valores distintos tiene y un ejemplo de su contenido
columnas_texto = df.select_dtypes(exclude=['number', 'bool']).columns

for columna in columnas_texto:
    ejemplo = df[columna].iloc[0]
    print(f'{columna:<20} | valores distintos: {df[columna].nunique():>4} | ejemplo: {ejemplo}')

fecha                | valores distintos:  546 | ejemplo: 2025-01-01
provincia            | valores distintos:   10 | ejemplo: CABA
fecha_alta_cliente   | valores distintos:  755 | ejemplo: 2025-01-01
canal                | valores distintos:    4 | ejemplo: Instagram Shop
categoria            | valores distintos:    5 | ejemplo: Indumentaria
producto             | valores distintos:   60 | ejemplo: Vestido midi
medio_pago           | valores distintos:    4 | ejemplo: Tarjeta de débito


**¿Son correctos los tipos de datos?** Casi todos:

- **Ninguna variable numérica se leyó como texto.** Precios, costos, montos, cantidades, comisiones y descuentos llegaron como `int64` o `float64`.
- **Dos fechas se leyeron como texto (`object`): `fecha` y `fecha_alta_cliente`.** Como texto no se pueden restar fechas, filtrar por período ni agrupar por mes. Las convierto con `pd.to_datetime` en el punto 2.4.
- **`recompra_90d` y `dias_desde_ultima_compra` aparecen como `float64`** aunque conceptualmente son enteros (0/1 y días). No es un error: en pandas una columna `int` no puede guardar `NaN`, y ambas tienen nulos (bloque 3).
- `provincia`, `canal`, `categoria`, `producto` y `medio_pago` son texto de verdad: variables categóricas con 10, 4, 5, 60 y 4 valores distintos.
- `es_evento` llegó como `bool`, correcto.
- `id_venta` e `id_cliente` son `int64`, pero son identificadores, no cantidades.

### 2.3 Foto estadística de las variables numéricas

In [6]:
# describe() resume las columnas numéricas. Lo transpongo (.T) para leer
# una variable por fila, y redondeo a 2 decimales para que sea legible
display(df.describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
id_venta,7617.0,3809.00,2198.98,1.00,1905.00,3809.00,5713.00,7617.00
id_cliente,7617.0,3134.37,1827.50,1.00,1549.00,3099.00,4743.00,6300.00
antiguedad_dias,7617.0,99.72,146.18,0.00,0.00,20.00,156.00,903.00
compras_previas,7617.0,1.93,3.29,0.00,0.00,1.00,2.00,28.00
dias_desde_ultima_compra,3863.0,67.14,51.56,3.00,32.00,52.00,86.00,504.00
gasto_previo_acum,7617.0,288504.26,534734.39,0.00,0.00,18570.51,339613.91,4994735.32
comision_canal_pct,7617.0,0.09,0.05,0.03,0.06,0.08,0.14,0.17
precio_unitario,7617.0,113403.86,86503.49,12179.90,47776.05,85030.33,150423.87,394586.64
costo_unitario,7617.0,72161.04,62657.90,6194.50,26086.29,48652.15,94931.15,284047.13
cantidad,7617.0,1.68,0.78,1.00,1.00,2.00,2.00,4.00


**Hallazgos y anomalías del `describe()`**

1. **`monto` es muy asimétrico, con outliers a la derecha.** La media (\$174.946) está 43% por encima de la mediana (\$122.352) y el máximo (\$1.344.061) es 11 veces la mediana: 4 unidades de un producto de Tecnología de \$353.700 cada una. No son errores de carga sino compras grandes reales, así que **no se eliminan**. Para describir el ticket típico conviene la mediana, no la media.
2. **`gasto_previo_acum` tiene un desvío casi del doble de su media** (\$534.734 contra \$288.504; coeficiente de variación de 1,85). El primer cuartil es 0 y la mediana apenas \$18.571 porque el 49,3% de las filas son primeras compras, donde el gasto previo es 0. **Esos ceros son reales, no datos faltantes.** En el otro extremo, unos pocos clientes muy leales acumulan hasta \$4,99 millones.
3. **Identificadores tratados como números.** `id_venta` e `id_cliente` aparecen con media y desvío, que no significan nada. Hay que excluirlos del análisis y del modelo: un ID nunca debe usarse como variable predictora.
4. **No hay valores imposibles.** Precios, costos y cantidades son siempre positivos; el descuento va de 0% a 25%; la comisión de 3% a 17% (el 17% es Mercado Libre después de la suba de octubre 2025); la recencia va de 3 a 504 días. En `cuotas`, la mediana y el tercer cuartil son 1: casi 8 de cada 10 compras se pagan en un pago, y el máximo de 12 corresponde a tarjeta de crédito (según el diccionario de datos).
5. **El target está balanceado, pero incompleto.** La media de `recompra_90d` es 0,42: el 42,1% de las compras tuvo recompra en 90 días, una proporción equilibrada entre clases. Su `count` es 6.483 sobre 7.617: faltan 1.134 valores (bloque 3).

### 2.4 Corrección de tipos: fechas

In [7]:
# Convierto las fechas de texto a datetime: habilita restar fechas, filtrar por período
# y agrupar por mes (lo necesito para el EDA y para respetar la ventana de 90 días del target)
df['fecha'] = pd.to_datetime(df['fecha'])
df['fecha_alta_cliente'] = pd.to_datetime(df['fecha_alta_cliente'])

print(df[['fecha', 'fecha_alta_cliente']].dtypes)
print(f'\nPeríodo de las compras: {df["fecha"].min().date()} a {df["fecha"].max().date()}')
print(f'Alta de cliente más antigua: {df["fecha_alta_cliente"].min().date()}')

fecha                 datetime64[ns]
fecha_alta_cliente    datetime64[ns]
dtype: object

Período de las compras: 2025-01-01 a 2026-06-30
Alta de cliente más antigua: 2024-01-01


Las dos fechas quedan como `datetime64`. Las compras van de enero 2025 a junio 2026, pero hay clientes dados de alta en 2024: el generador simula historia previa, y por eso existen compras de enero 2025 con `compras_previas` mayor a 0.

## 3. Diagnóstico de datos faltantes

In [8]:
# ==========================================
# BLOQUE 3: DIAGNÓSTICO DE DATOS FALTANTES
# ==========================================
# isnull() devuelve una máscara True/False: su suma cuenta los nulos
# y su promedio (x 100) da el porcentaje exacto de nulos por columna
reporte_nulos = pd.DataFrame({
    'nulos': df.isnull().sum(),
    '% nulos': (df.isnull().mean() * 100).round(2)
})

# Ranking de mayor a menor para priorizar qué columnas limpiar
reporte_nulos = reporte_nulos.sort_values('% nulos', ascending=False)
display(reporte_nulos)

columnas_con_nulos = reporte_nulos[reporte_nulos['nulos'] > 0]
print(f'Columnas con nulos: {len(columnas_con_nulos)} de {df.shape[1]}')

,nulos,% nulos
dias_desde_ultima_compra,3754,49.28
recompra_90d,1134,14.89
medio_pago,232,3.05
provincia,0,0.00
id_venta,0,0.00
fecha,0,0.00
id_cliente,0,0.00
compras_previas,0,0.00
antiguedad_dias,0,0.00
fecha_alta_cliente,0,0.00


Columnas con nulos: 3 de 23


Antes de decidir cómo limpiar, verifico el **origen** de cada nulo: no es lo mismo un dato que falta que un dato que no existe.

In [9]:
# Verificación 1: ¿los nulos de recencia son exactamente las primeras compras?
es_primera_compra = df['compras_previas'] == 0
recencia_nula = df['dias_desde_ultima_compra'].isnull()
print(f'Primeras compras (compras_previas == 0): {es_primera_compra.sum():,}')
print(f'Recencia nula:                           {recencia_nula.sum():,}')
print(f'¿Coinciden fila por fila?: {(es_primera_compra == recencia_nula).all()}')

# Verificación 2: ¿los nulos del target caen en la ventana truncada?
# Una compra posterior al 01/04/2026 no llega a tener 90 días de observación antes del 30/06/2026
target_nulo = df['recompra_90d'].isnull()
print(f'\nTarget nulo: compras del {df.loc[target_nulo, "fecha"].min().date()} '
      f'al {df.loc[target_nulo, "fecha"].max().date()}')
print('Valores del target en compras posteriores al 01/04/2026:')
print(df[df['fecha'] > '2026-04-01']['recompra_90d'].value_counts(dropna=False))

Primeras compras (compras_previas == 0): 3,754
Recencia nula:                           3,754
¿Coinciden fila por fila?: True

Target nulo: compras del 2026-04-02 al 2026-06-30
Valores del target en compras posteriores al 01/04/2026:
recompra_90d
NaN    1134
1.0     207
Name: count, dtype: int64


**Diagnóstico y decisión por columna**

| Columna | % nulos | Origen del nulo | Decisión para la limpieza |
|---|---|---|---|
| `dias_desde_ultima_compra` | 49,28% | **Estructural.** Coincide fila por fila con las 3.754 primeras compras: no existe una compra anterior desde la cual medir. | **No imputar** con media ni mediana: inventaría una recencia que no existe. La convierto en el segmento "Primera compra" con la función del bloque 4. |
| `recompra_90d` | 14,89% | **Censura.** Compras posteriores al 01/04/2026 sin 90 días completos de observación y sin recompra registrada. El resultado es *desconocido*, no negativo. | **Nunca rellenar con 0.** Se excluyen del entrenamiento del modelo: falta justo la etiqueta que se quiere predecir. |
| `medio_pago` | 3,05% | **Faltante real.** Dato no informado en 232 compras. | No borrar filas (son compras válidas con target). Imputar con la categoría `"No informado"`. |
| Las otras 20 | 0% | — | — |

**Trampa detectada.** Después del 01/04/2026 solo conservan target las compras que *sí* recompraron: 207 filas, todas con valor 1. Cualquier análisis de la recompra en el tiempo tiene que limitarse a compras con ventana completa (`fecha` ≤ 01/04/2026); si no, los últimos meses muestran una recompra artificialmente alta.

## 4. Lógica algorítmica y funciones personalizadas

### 4.1 Función formal (`def`): segmento de recencia

**Regla de negocio.** La recencia (días desde la compra anterior) es la señal más directa de si un cliente sigue "vivo". Uso cortes alineados con la ventana de 90 días del target, y cada segmento se traduce en una acción concreta de retención:

| Segmento | Regla | Acción de retención sugerida |
|---|---|---|
| Primera compra | sin compra previa (`NaN`) | Campaña de segunda compra |
| Activo | hasta 30 días | No regalar margen: probablemente vuelve solo |
| Tibio | 31 a 90 días | Recordatorio o remarketing |
| En riesgo | 91 a 180 días | Incentivo: envío gratis o descuento |
| Dormido | más de 180 días | Reactivación de bajo costo |

Además, la función resuelve el 49% de nulos de `dias_desde_ultima_compra` sin inventar datos: el `NaN` se convierte en información ("es su primera compra").

In [10]:
# ==========================================
# BLOQUE 4: FUNCIONES PERSONALIZADAS
# ==========================================
def segmentar_recencia(dias):
    """
    Clasifica una compra según los días transcurridos desde la compra anterior del cliente.

    Parámetro:
        dias (float): valor de 'dias_desde_ultima_compra'. Es NaN si es la primera compra.
    Retorna:
        str: segmento de recencia, asociado a una acción de retención.
    """
    # El NaN no es un error: es una primera compra (verificado en el bloque 3)
    if pd.isnull(dias):
        return 'Primera compra'
    elif dias <= 30:
        return 'Activo (≤30 días)'
    elif dias <= 90:
        return 'Tibio (31-90 días)'
    elif dias <= 180:
        return 'En riesgo (91-180 días)'
    else:
        return 'Dormido (+180 días)'


# Pruebo la función con casos borde antes de aplicarla a 7.617 filas
casos_prueba = [float('nan'), 15, 30, 31, 90, 91, 180, 504]
for caso in casos_prueba:
    print(f'{caso:>6} días -> {segmentar_recencia(caso)}')

   nan días -> Primera compra
    15 días -> Activo (≤30 días)
    30 días -> Activo (≤30 días)
    31 días -> Tibio (31-90 días)
    90 días -> Tibio (31-90 días)
    91 días -> En riesgo (91-180 días)
   180 días -> En riesgo (91-180 días)
   504 días -> Dormido (+180 días)


In [11]:
# Aplico la función a cada valor de la columna y guardo el resultado en una columna nueva
df['segmento_recencia'] = df['dias_desde_ultima_compra'].apply(segmentar_recencia)

display(df['segmento_recencia'].value_counts())

segmento_recencia
Primera compra             3754
Tibio (31-90 días)         2058
Activo (≤30 días)           906
En riesgo (91-180 días)     748
Dormido (+180 días)         151
Name: count, dtype: int64

In [12]:
# ¿La regla tiene sentido de negocio? Tasa de recompra por segmento.
# Uso solo compras con ventana completa de 90 días (fecha <= 01/04/2026)
# para no caer en la trampa de censura detectada en el bloque 3
ventana_completa = df[df['fecha'] <= '2026-04-01']

recompra_por_segmento = (
    ventana_completa
    .groupby('segmento_recencia')['recompra_90d']
    .agg(['count', 'mean'])
    .sort_values('mean', ascending=False)
)
recompra_por_segmento['mean'] = (recompra_por_segmento['mean'] * 100).round(1)
recompra_por_segmento.columns = ['compras', '% recompra 90d']

display(recompra_por_segmento)

,compras,% recompra 90d
segmento_recencia,,
Activo (≤30 días),780,80.9
Tibio (31-90 días),1722,54.3
En riesgo (91-180 días),612,32.7
Primera compra,3042,24.0
Dormido (+180 días),120,22.5


**Lectura.** La segmentación ordena la recompra de forma consistente: **Activo 80,9% → Tibio 54,3% → En riesgo 32,7% → Dormido 22,5%**. El dato más interesante es que *Primera compra* (24,0%) queda casi igual que *Dormido*: un cliente nuevo tiene la misma probabilidad de volver que uno que no compra hace más de seis meses. La segunda compra es el momento crítico de la retención. Es un adelanto de la hipótesis de recencia, que se valida formalmente en el EDA.

### 4.2 Funciones express (`lambda`)

**a) Transformación matemática: monto en pesos constantes.** La inflación acumulada del período es de 73%, así que comparar montos de enero 2025 con junio 2026 en pesos corrientes es comparar unidades distintas. Deflacto `monto` con el índice de precios del mes de cada compra y lo expreso en pesos de enero 2025.

In [13]:
# Índice de enero 2025 (el más bajo del período, porque el índice es acumulado) = base de los pesos constantes
INDICE_BASE = df['indice_inflacion'].min()

# La lambda recibe la fila completa (axis=1) porque usa dos columnas a la vez
df['monto_real'] = df.apply(
    lambda fila: round(fila['monto'] / fila['indice_inflacion'] * INDICE_BASE, 2),
    axis=1
)

display(df[['fecha', 'monto', 'indice_inflacion', 'monto_real']].tail())

,fecha,monto,indice_inflacion,monto_real
7612,2026-06-30,162771.59,1.8089,94033.01
7613,2026-06-30,135153.89,1.8089,78078.29
7614,2026-06-30,460364.97,1.8089,265952.45
7615,2026-06-30,368107.03,1.8089,212655.12
7616,2026-06-30,108179.65,1.8089,62495.29


In [14]:
# ¿Cuánto cambia la lectura? Ticket mediano de enero 2025 contra junio 2026
ene_2025 = df[df['fecha'] < '2025-02-01']
jun_2026 = df[df['fecha'] >= '2026-06-01']

var_nominal = (jun_2026['monto'].median() / ene_2025['monto'].median() - 1) * 100
var_real = (jun_2026['monto_real'].median() / ene_2025['monto_real'].median() - 1) * 100

print(f'Variación del ticket mediano en pesos corrientes:  {var_nominal:+.1f}%')
print(f'Variación del ticket mediano en pesos constantes: {var_real:+.1f}%')

Variación del ticket mediano en pesos corrientes:  +37.3%
Variación del ticket mediano en pesos constantes: -20.7%


**Lectura.** En pesos corrientes el ticket mediano parece crecer 37%; en pesos constantes **cae 21%**. Sin deflactar, el análisis contaría una historia de crecimiento que no existe. Por eso, en el EDA todas las comparaciones de montos entre períodos se hacen con `monto_real` (queda por analizar si la caída se explica por la mezcla de productos o de canales).

**b) Transformación de texto: código de canal.**

In [15]:
# Normalizo el canal a un código sin mayúsculas ni espacios ("Mercado Libre" -> "mercado_libre"):
# sirve como nombre de columna cuando codifique el canal para el modelo
# y evita errores de tipeo al filtrar ("Mercado libre" no es igual a "Mercado Libre")
df['canal_codigo'] = df['canal'].apply(lambda texto: texto.strip().lower().replace(' ', '_'))

display(df[['canal', 'canal_codigo']].drop_duplicates())

,canal,canal_codigo
0,Instagram Shop,instagram_shop
3,Tienda propia,tienda_propia
4,Mercado Libre,mercado_libre
11,WhatsApp,whatsapp


### 4.3 Resultado de las transformaciones

In [16]:
columnas_nuevas = ['segmento_recencia', 'monto_real', 'canal_codigo']
print(f'Dimensiones finales: {df.shape} -> {len(columnas_nuevas)} columnas nuevas: {columnas_nuevas}')

display(df[['fecha', 'id_cliente', 'dias_desde_ultima_compra', 'segmento_recencia',
            'canal', 'canal_codigo', 'monto', 'monto_real', 'recompra_90d']].head())

Dimensiones finales: (7617, 26) -> 3 columnas nuevas: ['segmento_recencia', 'monto_real', 'canal_codigo']


,fecha,id_cliente,dias_desde_ultima_compra,segmento_recencia,canal,canal_codigo,monto,monto_real,recompra_90d
0,2025-01-01,335,NaN,Primera compra,Instagram Shop,instagram_shop,56018.26,56018.26,1.0
1,2025-01-01,455,NaN,Primera compra,Instagram Shop,instagram_shop,40803.13,40803.13,0.0
2,2025-01-01,1185,39.0,Tibio (31-90 días),Instagram Shop,instagram_shop,60874.23,60874.23,1.0
3,2025-01-01,1913,17.0,Activo (≤30 días),Tienda propia,tienda_propia,54863.95,54863.95,1.0
4,2025-01-01,2286,NaN,Primera compra,Mercado Libre,mercado_libre,56679.43,56679.43,0.0


## 5. Conclusiones y próximos pasos

**Lo que dejó el diagnóstico**

- **El dataset está sano:** 7.617 × 23, sin duplicados, sin valores imposibles y con todas las variables numéricas bien tipadas. La única corrección de tipos fue convertir las dos fechas a `datetime`.
- **Hay nulos en 3 columnas y ninguno es "basura":** uno estructural (recencia en primeras compras, 49,3%), uno por censura del target (14,9%) y un faltante real chico (`medio_pago`, 3,1%). Cada uno lleva un tratamiento distinto.
- **`monto` y `gasto_previo_acum` son muy asimétricos:** se resumen con la mediana, y sus outliers son negocio real.
- **La recencia ya muestra señal:** la recompra va de 80,9% en clientes activos a 22,5% en dormidos, y la primera compra retiene tan poco como un cliente dormido.
- **La inflación invierte la lectura del ticket:** +37% en pesos corrientes, −21% en pesos constantes.

**Próximos pasos**

1. Limpieza: imputar `medio_pago` con "No informado" y separar las filas con target desconocido.
2. EDA con visualizaciones para validar las seis hipótesis del proyecto: recurrencia, eventos, recencia, canal, categoría y ticket real.
3. Preparar las features para el primer modelo de clasificación de `recompra_90d`.